# 01 — Data Loading & Quality Audit

**Airline Operations Intelligence Platform** · Notebook 1 of 10 · *runs locally*

## Purpose
Load the Kaggle 2015 flight dataset into PySpark, audit it honestly, and produce the
**ETL contract** that notebook `02` implements.

This notebook does not clean anything. Its job is to find every defect and write down
the rule that fixes it, so cleaning decisions are evidence-based rather than guessed.

## Environment
Local Spark — no Colab, no Drive. Verified working on this machine:

| Component | Version |
|---|---|
| Java | 21.0.2 LTS |
| Python | 3.12.1 |
| PySpark | 4.0.0 |
| Machine | 8 GB RAM, 8 cores → `local[6]`, 3 GB driver |

## Outputs
| Output | Path | Consumed by |
|---|---|---|
| Raw Parquet (partitioned by month) | `data/raw/flights_raw.parquet` | `02` |
| Frozen schema | `data/raw/flights_schema.json` | `02`, `09` |
| Findings table | §12 below | `02`, project report |

## Syllabus coverage
- **Unit 1 — Volume:** 5.8M records. **Veracity:** §5–§8 quantify every defect.
- **Unit 2 — Columnar storage:** §11 measures Parquet vs CSV on a partial-column scan.
- **Unit 4 — Lazy evaluation & DAG:** §4 separates transformation build time from action time.

---
## 1. Session setup

Spark configuration lives in `src/config.py` so all ten notebooks stay consistent.

In [ ]:
import sys, time, json
sys.path.insert(0, "../src")

from config import build_spark, PATHS

spark = build_spark("01-loading")

print("Spark      :", spark.version)
print("Master     :", spark.sparkContext.master)
print("Cores      :", spark.sparkContext.defaultParallelism)
print("Spark UI   :", spark.sparkContext.uiWebUrl, " <- open this to watch jobs and DAGs")

### Spark architecture in local mode (Unit 4)

Running `local[6]` means the **driver** and **executors** are threads inside one JVM,
with 6 task slots. The cluster manager is a no-op. Nothing about the code changes on a
real cluster — only the master URL does. That portability is the point of the Spark
programming model: the same DAG is scheduled locally or across 100 machines.

---
## 2. Load the three CSVs

`inferSchema=True` costs a **full extra pass** over the 592 MB file. We pay it once here
to discover the real types, then freeze the schema in §10 so later notebooks load in
a single pass.

In [ ]:
t0 = time.time()
flights  = spark.read.csv(str(PATHS["flights_csv"]),  header=True, inferSchema=True)
airlines = spark.read.csv(str(PATHS["airlines_csv"]), header=True, inferSchema=True)
airports = spark.read.csv(str(PATHS["airports_csv"]), header=True, inferSchema=True)
print(f"Load + schema inference: {time.time()-t0:.1f}s")

In [ ]:
t0 = time.time()
N = flights.count()          # ACTION -> triggers the DAG
print(f"flights  : {N:>9,} rows x {len(flights.columns):>2} cols   ({time.time()-t0:.1f}s)")
print(f"airlines : {airlines.count():>9,} rows x {len(airlines.columns):>2} cols")
print(f"airports : {airports.count():>9,} rows x {len(airports.columns):>2} cols")

In [ ]:
flights.printSchema()

In [ ]:
flights.show(5, truncate=False)

In [ ]:
airlines.show(20, truncate=False)

In [ ]:
airports.show(8, truncate=False)
airports.printSchema()

---
## 3. Lazy evaluation and the DAG

Unit 4 requires evidence that transformations are lazy and only actions execute.
Time the two halves separately and the difference is stark.

In [ ]:
from pyspark.sql import functions as F

t0 = time.time()
chain = (flights
         .filter(F.col("CANCELLED") == 0)
         .filter(F.col("DEPARTURE_DELAY") > 15)
         .select("AIRLINE", "ORIGIN_AIRPORT", "DEPARTURE_DELAY")
         .groupBy("AIRLINE").count())
t_build = time.time() - t0

t0 = time.time()
_ = chain.collect()          # ACTION
t_exec = time.time() - t0

print(f"Build 4 transformations : {t_build:.4f}s   <- no computation happened")
print(f"collect() action        : {t_exec:.2f}s   <- whole DAG ran here")
print(f"Ratio                   : {t_exec/max(t_build,1e-6):,.0f}x")

In [ ]:
# Catalyst's physical plan. Look for PushedFilters (predicate pushdown) and Exchange (shuffle).
chain.explain(mode="formatted")

---
## 4. Missingness — and whether it is *structural*

The question is not "which columns have nulls" but "is each null explainable?".
A null that encodes a fact (flight was cancelled, so it has no arrival time) must be
preserved. A null that is corruption must be repaired. Filling the first kind with `0`
silently corrupts every downstream average.

In [ ]:
def missingness(df, total):
    exprs = [F.round(100.0 * F.count(F.when(F.col(col).isNull(), col)) / F.lit(total), 2).alias(col)
             for col in df.columns]
    return sorted(df.select(*exprs).collect()[0].asDict().items(), key=lambda kv: -kv[1])

print(f"{'COLUMN':<26}{'% NULL':>9}")
print("-" * 35)
for col, pct in missingness(flights, N):
    print(f"{col:<26}{pct:>8.2f}%" + ("   <- explain this" if 0 < pct < 99 else ""))

Three hypotheses to test before treating any of these as corruption:

1. Arrival/departure columns are null **because the flight was cancelled**.
2. The five `*_DELAY` cause columns are null **because the flight arrived on time** —
   the DOT only attributes causes to flights 15+ minutes late.
3. `CANCELLATION_REASON` is null **because the flight was not cancelled**.

In [ ]:
# Hypotheses 1 and 3.
flights.groupBy("CANCELLED").agg(
    F.count("*").alias("flights"),
    F.count(F.when(F.col("DEPARTURE_DELAY").isNull(), 1)).alias("null_dep_delay"),
    F.count(F.when(F.col("ARRIVAL_DELAY").isNull(),   1)).alias("null_arr_delay"),
    F.count(F.when(F.col("AIR_TIME").isNull(),        1)).alias("null_air_time"),
    F.count(F.when(F.col("CANCELLATION_REASON").isNotNull(), 1)).alias("has_cancel_reason"),
).show()

In [ ]:
# Hypothesis 2.
CAUSES = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
          "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

(flights
 .filter(F.col("CANCELLED") == 0)
 .withColumn("bucket", F.when(F.col("ARRIVAL_DELAY") >= 15, "arrived 15+ late")
                        .otherwise("on time / <15"))
 .groupBy("bucket")
 .agg(F.count("*").alias("flights"),
      *[F.count(F.when(F.col(col).isNotNull(), 1)).alias(col.split("_")[0].lower())
        for col in CAUSES])
 .show(truncate=False))

In [ ]:
# Diverted flights are a third structural case - they depart but never arrive as scheduled.
flights.groupBy("DIVERTED").agg(
    F.count("*").alias("flights"),
    F.count(F.when(F.col("ARRIVAL_DELAY").isNull(), 1)).alias("null_arr_delay"),
).show()

---
## 5. Duplicates

Exact-row duplicates and business-key collisions are different problems with different fixes.
A flight is identified by date + airline + flight number + origin + scheduled departure.

In [ ]:
KEY = ["YEAR","MONTH","DAY","AIRLINE","FLIGHT_NUMBER","ORIGIN_AIRPORT","SCHEDULED_DEPARTURE"]

n_exact = flights.distinct().count()
n_key   = flights.select(*KEY).distinct().count()

print(f"Total rows              : {N:,}")
print(f"Distinct full rows      : {n_exact:,}   -> {N-n_exact:,} exact duplicates")
print(f"Distinct business keys  : {n_key:,}   -> {N-n_key:,} key collisions")

In [ ]:
# Are collisions real duplicates, or genuinely different flights sharing a key?
collisions = flights.groupBy(*KEY).count().filter(F.col("count") > 1)
print(f"Keys appearing more than once: {collisions.count():,}")
collisions.orderBy(F.desc("count")).show(5, truncate=False)

---
## 6. The airport-code defect

**The most damaging defect in this dataset.** For part of the year, `ORIGIN_AIRPORT` and
`DESTINATION_AIRPORT` hold 5-digit numeric **DOT codes** instead of 3-letter IATA codes.
`airports.csv` is keyed on IATA only, so an inner join silently discards those rows —
no error, no warning, just a whole month missing from every airport and route metric.

In [ ]:
flights.select(F.length("ORIGIN_AIRPORT").alias("code_length")) \
       .groupBy("code_length").count().orderBy("code_length").show()

In [ ]:
# Which months are affected?
(flights
 .withColumn("code_type", F.when(F.length("ORIGIN_AIRPORT") == 3, "IATA (3-char)")
                           .otherwise("DOT numeric (5-digit)"))
 .groupBy("MONTH").pivot("code_type").count()
 .orderBy("MONTH").show(12, truncate=False))

In [ ]:
# Quantify what a naive inner join would destroy.
iata = airports.select(F.col("IATA_CODE").alias("code"))
matched = flights.select(F.col("ORIGIN_AIRPORT").alias("code")).join(iata, "code", "inner").count()

print(f"Flights whose ORIGIN_AIRPORT resolves in airports.csv : {matched:,}")
print(f"Flights a naive inner join would DROP                 : {N-matched:,}  ({100*(N-matched)/N:.1f}%)")
print()
print("ETL RULE (notebook 02): translate DOT numeric -> IATA before any join.")
print("Then LEFT join and assert the unmatched count is zero.")

In [ ]:
# Does airports.csv itself have gaps? Null lat/lon breaks the dashboard map.
airports.select(
    F.count("*").alias("airports"),
    F.countDistinct("IATA_CODE").alias("distinct_iata"),
    F.count(F.when(F.col("LATITUDE").isNull(), 1)).alias("null_lat"),
    F.count(F.when(F.col("LONGITUDE").isNull(), 1)).alias("null_lon"),
).show()

airports.filter(F.col("LATITUDE").isNull() | F.col("LONGITUDE").isNull()).show(truncate=False)

---
## 7. Impossible and extreme values

Two traps here:

- Time columns are **`HHMM` integers** (`1435` = 14:35), not minutes. `2400` is legal and
  means midnight. Treating them as numbers makes arithmetic silently wrong.
- Delays are **signed**. Negative means the flight was early — valid data, not an error.
  Clipping negatives to zero would inflate every average.

In [ ]:
flights.select("DEPARTURE_DELAY","ARRIVAL_DELAY","DISTANCE","AIR_TIME",
               "TAXI_OUT","TAXI_IN","SCHEDULED_TIME","ELAPSED_TIME").describe().show()

In [ ]:
for col in ["SCHEDULED_DEPARTURE","DEPARTURE_TIME","SCHEDULED_ARRIVAL","ARRIVAL_TIME"]:
    lo, hi = flights.select(F.min(col), F.max(col)).first()
    out_of_range = flights.filter(F.col(col).isNotNull() & ((F.col(col) < 0) | (F.col(col) > 2400))).count()
    bad_minutes  = flights.filter(F.col(col).isNotNull() & ((F.col(col) % 100) >= 60)).count()
    at_2400      = flights.filter(F.col(col) == 2400).count()
    print(f"{col:<21} range=[{lo},{hi}]  out_of_range={out_of_range:,}  mins>=60={bad_minutes:,}  ==2400={at_2400:,}")

In [ ]:
# Inspect the extreme tail before deciding whether to cap anything.
(flights.filter(F.col("DEPARTURE_DELAY").isNotNull())
        .select("MONTH","DAY","AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT",
                "DEPARTURE_DELAY","ARRIVAL_DELAY","CANCELLED","DIVERTED")
        .orderBy(F.desc("DEPARTURE_DELAY")).show(10, truncate=False))

In [ ]:
# Class balance -- this sets the ML baseline in notebook 06.
early  = flights.filter(F.col("DEPARTURE_DELAY") < 0).count()
ontime = flights.filter((F.col("DEPARTURE_DELAY") >= 0) & (F.col("DEPARTURE_DELAY") <= 15)).count()
late   = flights.filter(F.col("DEPARTURE_DELAY") > 15).count()
known  = early + ontime + late

print(f"Early   (<0 min)   : {early:>9,}  ({100*early/known:5.1f}%)")
print(f"On-time (0-15 min) : {ontime:>9,}  ({100*ontime/known:5.1f}%)")
print(f"Delayed (>15 min)  : {late:>9,}  ({100*late/known:5.1f}%)")
print()
print(f"A model predicting 'never delayed' scores {100*(early+ontime)/known:.1f}% accuracy")
print("and is useless. Notebook 06 must report precision/recall/F1, not accuracy.")

---
## 8. Cardinality and referential integrity

Confirms join keys for notebook 02 and sizes the categorical features for notebook 06 —
a high-cardinality column one-hot encoded becomes thousands of columns.

In [ ]:
for col in ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT","TAIL_NUMBER",
            "CANCELLATION_REASON","MONTH","DAY_OF_WEEK"]:
    print(f"{col:<22} distinct = {flights.select(col).distinct().count():>7,}")

In [ ]:
in_flights = {r[0] for r in flights.select("AIRLINE").distinct().collect()}
in_lookup  = {r[0] for r in airlines.select("IATA_CODE").collect()}

print("In flights but missing from airlines.csv :", in_flights - in_lookup or "none  <- good")
print("In airlines.csv but never flown          :", in_lookup - in_flights or "none  <- good")

In [ ]:
# A=Carrier  B=Weather  C=National Air System  D=Security
flights.filter(F.col("CANCELLED") == 1).groupBy("CANCELLATION_REASON").count() \
       .orderBy(F.desc("count")).show()

cancelled = flights.filter(F.col("CANCELLED") == 1).count()
diverted  = flights.filter(F.col("DIVERTED")  == 1).count()
print(f"Cancelled : {cancelled:,}  ({100*cancelled/N:.2f}%)")
print(f"Diverted  : {diverted:,}  ({100*diverted/N:.2f}%)")

---
## 9. Freeze the schema

Now that inference has revealed the true types, write them out. Notebooks 02+ load with
an explicit schema in one pass instead of two. The speedup is measured below.

In [ ]:
PATHS["schema"].write_text(flights.schema.json())
print("Wrote", PATHS["schema"])

In [ ]:
from pyspark.sql.types import StructType

FLIGHTS_SCHEMA = StructType.fromJson(json.loads(PATHS["schema"].read_text()))

t0 = time.time(); spark.read.csv(str(PATHS["flights_csv"]), header=True, schema=FLIGHTS_SCHEMA).count()
t_explicit = time.time() - t0

t0 = time.time(); spark.read.csv(str(PATHS["flights_csv"]), header=True, inferSchema=True).count()
t_infer = time.time() - t0

print(f"Explicit schema : {t_explicit:5.1f}s")
print(f"inferSchema     : {t_infer:5.1f}s")
print(f"Speedup         : {t_infer/t_explicit:5.2f}x")

---
## 10. Persist as Parquet

Parquet is the Hadoop-ecosystem **columnar** format (Unit 2). It stores data column-wise
with per-column compression and an embedded schema, so a query touching 3 of 31 columns
reads only those 3. Partitioning by `MONTH` additionally lets Spark skip whole directories.

Both claims are measured below rather than asserted.

In [ ]:
t0 = time.time()
flights.write.mode("overwrite").partitionBy("MONTH").parquet(str(PATHS["raw"] / "flights_raw.parquet"))
print(f"flights -> Parquet : {time.time()-t0:.1f}s")

airlines.write.mode("overwrite").parquet(str(PATHS["raw"] / "airlines_raw.parquet"))
airports.write.mode("overwrite").parquet(str(PATHS["raw"] / "airports_raw.parquet"))
print("airlines, airports -> Parquet : done")

In [ ]:
import subprocess
for label, path in [("CSV    ", PATHS["flights_csv"]), ("Parquet", PATHS["raw"] / "flights_raw.parquet")]:
    size = subprocess.run(["du","-sh",str(path)], capture_output=True, text=True).stdout.split()[0]
    print(f"{label} : {size}")

In [ ]:
# Columnar advantage: scan 3 of 31 columns.
t0 = time.time()
(spark.read.csv(str(PATHS["flights_csv"]), header=True, schema=FLIGHTS_SCHEMA)
      .select("AIRLINE","DEPARTURE_DELAY","DISTANCE").count())
t_csv = time.time() - t0

t0 = time.time()
(spark.read.parquet(str(PATHS["raw"] / "flights_raw.parquet"))
      .select("AIRLINE","DEPARTURE_DELAY","DISTANCE").count())
t_pq = time.time() - t0

print(f"3-column scan, CSV     : {t_csv:5.1f}s")
print(f"3-column scan, Parquet : {t_pq:5.1f}s")
print(f"Speedup                : {t_csv/t_pq:5.2f}x")

In [ ]:
# Partition pruning: filtering on MONTH should read a single partition directory.
(spark.read.parquet(str(PATHS["raw"] / "flights_raw.parquet"))
      .filter(F.col("MONTH") == 6).select("AIRLINE").explain(mode="simple"))

---
## 11. Findings — the ETL contract for notebook 02

**All values below are measured from an actual run on this machine** (5,819,079 rows,
local Spark 4.0, ~3 min end to end). Each row is a defect and the rule notebook `02` must implement.

| # | Defect | Measured | Rule for notebook 02 |
|---|---|---|---|
| 1 | Airport codes are DOT numeric, **October only** | **486,165 rows (8.4%)** — month 10 is 100% numeric, all other months 100% IATA | Translate DOT→IATA **before** any join. LEFT join, then assert unmatched = 0 |
| 2 | `ARRIVAL_DELAY` / `AIR_TIME` null when cancelled **or diverted** | 105,071 nulls = 89,884 cancelled + 15,187 diverted (1.81%) | Structural. Exclude both from delay stats. Never fill with 0 |
| 3 | Delay-cause columns null unless arrival ≥ 15 late | 81.72% null. Populated for **exactly** 1,063,439 late arrivals, 0 otherwise | Structural. Fill 0 **only** within the late subset |
| 4 | `CANCELLATION_REASON` null unless cancelled | 98.46% null; present for all 89,884 cancelled | Structural. Decode A/B/C/D to labels |
| 5 | Duplicate business keys | **0 exact duplicates; 1 key collision** (AA803 STT 2015-08-29 14:35) | Dedupe on the 7-column key. Cheap, and guards against re-runs |
| 6 | Negative delays = early departures | **57.2%** of flights depart early | Valid signal. Do **not** clip to zero |
| 7 | Times are `HHMM` ints | Range [0, 2400] confirmed; no minutes ≥ 60 | Convert to minutes-since-midnight; map 2400 → 0 |
| 8 | Class imbalance on ML target | **17.8% delayed >15 min** → 82.2% baseline accuracy | Notebook 06: class weights, stratified split, report F1 not accuracy |
| 9 | `TAIL_NUMBER` nulls | 0.25% | Not a feature. Ignore |
| 10 | Airports missing lat/lon | **3 of 322**: ECP, PBG, UST | Drop from map page only, never from metrics |
| 11 | Cancelled *after* pushback | 89,884 cancelled but only 86,153 lack `DEPARTURE_DELAY` → **3,731 departed then cancelled** | `CANCELLED = 1` does not imply "never left the gate". Filter on the delay column, not the flag alone |

### Measured performance (Unit 2 / Unit 4 evidence)

| Measurement | Result |
|---|---|
| Lazy vs action | build 4 transformations 0.13s → `collect()` 1.69s (**13×**) |
| Explicit schema vs `inferSchema` | 0.5s vs 3.9s (**8.2× faster**) |
| Storage: CSV → Parquet | 565 MB → **144 MB** (3.9× smaller) |
| 3-of-31-column scan, CSV vs Parquet | 0.6s vs 0.2s (**2.7× faster**) |
| Full write to partitioned Parquet | 8.8s |

### Volume (Unit 1)
5,819,079 flights × 31 columns · 565 MB CSV · 14 airlines · 322 airports ·
1.54% cancelled · 0.26% diverted.

### Next
`02_data_cleaning_etl.ipynb` — implement rules 1–11, write curated Parquet to
`data/curated/`, and assert a row-count contract at every stage so silent data loss
becomes impossible.

In [ ]:
spark.stop()
print("Notebook 01 complete.")